In [6]:
# =============================================================================
# 1. IMPORT LIBRARY
# =============================================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist, squareform

from scipy.optimize import minimize

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from pykrige.ok import OrdinaryKriging

from pykrige.kriging_tools import write_asc_grid

import plotly.graph_objects as go

from plotly.subplots import make_subplots

import warnings

warnings.filterwarnings("ignore")


plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

In [9]:
# =============================================================================
# 2. DATA SPASIAL DENGAN STRUKTUR AUTOKORELASI
# =============================================================================


print("="*80)
print("TAHAP 1: DATA SPASIAL PENELITIAN")
print("="*80)

# membaca data 
data = pd.read_excel(
    "soil.xlsx"
)

print("\nInformasi Data")
print(data.info())

print("\n5 data awal")
print(data.head())

# koordinat spasial

coordinate_x = data["x"].values

coordinate_y = data["y"].values

observed_value = data["zinc"].values


print("\nStatistik Data Zinc")
print(
    pd.Series(observed_value).describe()
)

TAHAP 1: DATA SPASIAL PENELITIAN

Informasi Data
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   x       155 non-null    int64
 1   y       155 non-null    int64
 2   zinc    155 non-null    int64
dtypes: int64(3)
memory usage: 3.8 KB
None

5 data awal
        x       y  zinc
0  181072  333611  1022
1  181025  333558  1141
2  181165  333537   640
3  181298  333484   257
4  181307  333330   269

Statistik Data Zinc
count     155.000000
mean      469.716129
std       367.073788
min       113.000000
25%       198.000000
50%       326.000000
75%       674.500000
max      1839.000000
dtype: float64


In [13]:
# =============================================================================
# 3. SAMPLING DATA (DATA OBSERVASI)
# =============================================================================


print("="*80)
print("TAHAP 2: SAMPLING DATA")
print("="*80)


sample_data = data.sample(
    frac=0.8,
    random_state=10
)


validation_data = data.drop(
    sample_data.index
)


print(
    "Jumlah data training:",
    len(sample_data)
)


print(
    "Jumlah data validasi:",
    len(validation_data)
)


TAHAP 2: SAMPLING DATA
Jumlah data training: 124
Jumlah data validasi: 31


In [14]:
# =============================================================================
# 4. ANALISIS VARIOGRAM
# =============================================================================

print("="*80)
print("TAHAP 3: ANALISIS VARIOGRAM")
print("="*80)


# Membuat koordinat spasial
coords = np.column_stack(
    (
        sample_data["x"],
        sample_data["y"]
    )
)


# Nilai observasi zinc
values = sample_data["zinc"].values


print("Jumlah titik observasi:")
print(len(values))

distance_matrix = squareform(
    pdist(coords)
)


variogram_data = []


for i in range(len(values)):

    for j in range(i+1, len(values)):

        distance = distance_matrix[i,j]

        semivariance = (
            (values[i]-values[j])**2
        ) / 2


        variogram_data.append(
            [
                distance,
                semivariance
            ]
        )


variogram_df = pd.DataFrame(
    variogram_data,
    columns=[
        "distance",
        "semivariance"
    ]
)


print(variogram_df.head())

TAHAP 3: ANALISIS VARIOGRAM
Jumlah titik observasi:
124
      distance  semivariance
0  2951.312251     1099644.5
1  1122.748859      808992.0
2   739.649241     1205904.5
3  1866.590475      273060.5
4  1372.433241     1086338.0
